In [ ]:
# ============================================================
# PRODUCT QUANTIZATION ERROR:
# DECOMPOSITION, PERIODICITY AND RATIONAL MULTIPLIERS
# ============================================================
#
# This notebook studies the quantization of the product
#
#                   alpha x[n]
#
# when a finite-precision fixed-point implementation is used.
#
# The quantization process is modeled as
#
#                   x[n]
#                     |
#                     v
#                   Q(x[n])
#                     |
#                     v
#               alpha Q(x[n])
#                     |
#                     v
#               Q(alpha Q(x[n]))
#
# The total error is therefore
#
#       e[n]
#
#       = Q(alpha Q(x[n])) - alpha x[n]
#
#       = [Q(alpha Q(x[n])) - alpha Q(x[n])]
#         + alpha [Q(x[n]) - x[n]]
#
# or equivalently
#
#       e[n] = e_product[n] + alpha e_input[n].
#
#
# ============================================================
# ERROR COMPONENTS
# ============================================================
#
# The first term
#
#       e_product[n]
#
#       = Q(alpha Q(x[n])) - alpha Q(x[n])
#
# is the rounding error produced by quantizing the result of the
# multiplication.
#
# The second term
#
#       alpha e_input[n]
#
#       = alpha [Q(x[n]) - x[n]]
#
# represents the contribution of the original signal-quantization
# error after multiplication by alpha.
#
# Therefore, the total product error contains TWO distinct
# contributions.
#
#
# ============================================================
# ERROR BOUNDS
# ============================================================
#
# For rounding quantization with step
#
#                   Delta = 2^(-K),
#
# each individual quantization error satisfies approximately
#
#                   |e| <= Delta / 2
#
# when overflow is absent.
#
# Consequently,
#
#       |e_product| <= Delta / 2
#
# and
#
#       |alpha e_input| <= |alpha| Delta / 2.
#
# A conservative bound for the total error is therefore
#
#       |e_total| <= (1 + |alpha|) Delta / 2.
#
#
# ============================================================
# RATIONAL MULTIPLIERS
# ============================================================
#
# A particularly interesting situation appears when
#
#                   alpha = L / M
#
# where L and M are relatively prime integers.
#
# If the input has already been quantized,
#
#                   x = n Delta,
#
# then
#
#                   alpha x = (L/M) n Delta.
#
# After rounding to the nearest multiple of Delta,
#
#                   Q(alpha x) = k Delta
#
# and
#
#                   e = Q(alpha x) - alpha x
#
#                     = (kM - nL) Delta / M
#
#                     = ell Delta / M.
#
# The integer ell satisfies
#
#                   ell + nL = 0   (mod M).
#
# The same error pattern repeats after M samples:
#
#                   e[(n+M)Delta] = e[nDelta].
#
# Therefore, the rounding-error sequence is periodic with
# period M.
#
#
# ============================================================
# HOW TO USE THIS NOTEBOOK
# ============================================================
#
# The notebook contains four display modes.
#
#
# 1. TOTAL PRODUCT ERROR
#
#    Active controls:
#
#           alpha
#           K
#
#    The graph displays the total error
#
#           e_total(x).
#
#
# 2. ERROR DECOMPOSITION
#
#    Active controls:
#
#           alpha
#           K
#
#    The graph separately displays
#
#           e_product
#
#           alpha e_input
#
#           e_total.
#
#
# 3. RATIONAL MULTIPLIER PERIODICITY
#
#    Active controls:
#
#           L
#           M
#           Samples
#
#    The multiplier is
#
#           alpha = L/M
#
#    and the input consists of already-quantized values
#
#           x = n Delta.
#
#    The error sequence repeats every M samples.
#
#
# 4. ERROR VALUES OVER ONE PERIOD
#
#    Active controls:
#
#           L
#           M
#
#    Only one complete period
#
#           n = 0, 1, ..., M-1
#
#    is displayed.
#
#
# ============================================================
# WHAT WE EXPECT TO OBSERVE
# ============================================================
#
# TOTAL PRODUCT ERROR
#
# Increasing K decreases
#
#                   Delta = 2^(-K)
#
# and therefore reduces the absolute size of the product
# quantization error.
#
# Changing alpha alters both the multiplication itself and the
# contribution alpha e_input.
#
#
# ERROR DECOMPOSITION
#
# The total error satisfies
#
#       e_total = e_product + alpha e_input.
#
# Increasing |alpha| increases the contribution of
#
#       alpha e_input
#
# relative to the product-rounding contribution.
#
#
# RATIONAL MULTIPLIER PERIODICITY
#
# When
#
#                   alpha = L/M
#
# and gcd(L,M) = 1, the rounding error is periodic with period M.
#
# Vertical separators identify successive M-sample periods.
#
#
# ERROR VALUES OVER ONE PERIOD
#
# The error takes discrete values that are integer multiples of
#
#                   Delta / M.
#
# The one-period graph exposes this discrete structure directly.
#
#
# ============================================================
# IMPORTANT NOTE ABOUT NORMALIZATION
# ============================================================
#
# In the rational-multiplier plots the vertical axis is
#
#                   e / Delta.
#
# The real error DOES depend on K because
#
#                   Delta = 2^(-K).
#
# However, dividing the error by Delta removes this scale factor.
#
# Therefore, changing K does not change the SHAPE of the
# normalized rational-multiplier error curves.
#
# For this reason, the K slider is disabled in these plots.
#
# This does NOT mean that the physical quantization error is
# independent of K. Increasing K still decreases the absolute
# error because Delta becomes smaller.
#
#
# ============================================================
# WHY SOME CONTROLS BECOME DISABLED
# ============================================================
#
# Controls that do not affect the currently displayed quantity
# are automatically disabled.
#
# This prevents parameters belonging to the general-alpha model
# from being confused with parameters belonging to the rational
# alpha = L/M model.
#
#
# ============================================================
# IMPORTANT INTERPRETATION
# ============================================================
#
# The periodic structure demonstrated here shows that fixed-point
# multiplication error is NOT always equivalent to independent
# white noise.
#
# For rational multipliers and quantized input values, the
# rounding error can be strongly structured and periodic.
#
# The white-noise model must therefore be regarded as a
# statistical approximation whose validity depends on the
# signal and implementation conditions.
#
#
# ============================================================
# MODELING ASSUMPTIONS
# ============================================================
#
# 1. Fixed-point arithmetic is assumed.
#
# 2. Quantization is performed by rounding to the nearest
#    multiple of Delta.
#
# 3. Overflow is excluded.
#
# 4. For the rational-multiplier demonstrations,
#
#                   x = n Delta.
#
# 5. L/M is automatically reduced to lowest terms.
#
# ============================================================


%matplotlib inline

import math
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, IntSlider, RadioButtons, VBox, HBox, HTML, Layout, interactive
from IPython.display import display


# ------------------------------------------------------------
# Quantizer
# ------------------------------------------------------------

def round_quantizer(x, Delta):

    x = np.asarray(x)

    q = np.where(x >= 0, np.floor(x / Delta + 0.5), np.ceil(x / Delta - 0.5))

    return Delta * q


# ------------------------------------------------------------
# General product-error decomposition
# ------------------------------------------------------------

def product_error_components(x, alpha, Delta):

    qx = round_quantizer(x, Delta)

    product_before_final_quantization = alpha * qx

    q_product = round_quantizer(product_before_final_quantization, Delta)

    e_input = qx - x

    e_product = q_product - product_before_final_quantization

    e_input_contribution = alpha * e_input

    e_total = q_product - alpha * x

    return qx, q_product, e_input, e_product, e_input_contribution, e_total


# ------------------------------------------------------------
# Rational multiplier error sequence
# ------------------------------------------------------------

def rational_error_sequence(L, M, K, number_of_samples):

    divisor = math.gcd(L, M)

    Lr = L // divisor

    Mr = M // divisor

    alpha = Lr / Mr

    Delta = 2.0**(-K)

    n = np.arange(number_of_samples)

    x = n * Delta

    exact_product = alpha * x

    quantized_product = round_quantizer(exact_product, Delta)

    error = quantized_product - exact_product

    return n, error, Delta, alpha, Lr, Mr


# ------------------------------------------------------------
# One-period error sequence
# ------------------------------------------------------------

def one_period_error(L, M, K):

    divisor = math.gcd(L, M)

    Lr = L // divisor

    Mr = M // divisor

    Delta = 2.0**(-K)

    alpha = Lr / Mr

    n = np.arange(Mr)

    x = n * Delta

    exact_product = alpha * x

    quantized_product = round_quantizer(exact_product, Delta)

    error = quantized_product - exact_product

    integer_levels = np.rint(error * Mr / Delta).astype(int)

    return n, error, integer_levels, Delta, alpha, Lr, Mr


# ------------------------------------------------------------
# Style
# ------------------------------------------------------------

style_html = HTML("""
<style>

.pe-root {
    font-family: monospace;
    width: 960px;
    max-width: 960px;
}

.pe-description {
    font-size: 13px;
    line-height: 1.45;
    padding: 10px 12px;
    border: 1px solid #bfc7d5;
    border-left: 6px solid #4a6fa5;
    background: #f7f9fc;
    border-radius: 8px;
    margin-bottom: 8px;
    box-sizing: border-box;
}

.pe-observe {
    font-size: 12.5px;
    line-height: 1.45;
    padding: 10px 12px;
    border: 1px solid #c7d8c9;
    border-left: 6px solid #3c8a4e;
    background: #f7fbf7;
    border-radius: 8px;
    margin-top: 8px;
    box-sizing: border-box;
}

.pe-observe-title {
    font-size: 13px;
    font-weight: bold;
    color: #245c31;
    margin-bottom: 5px;
}

.pe-normalization {
    font-size: 12.5px;
    line-height: 1.45;
    padding: 9px 12px;
    border: 1px solid #d5c58a;
    border-left: 6px solid #b8860b;
    background: #fffaf0;
    border-radius: 8px;
    margin-top: 8px;
    box-sizing: border-box;
}

.pe-normalization-title {
    font-size: 13px;
    font-weight: bold;
    color: #8a6500;
    margin-bottom: 5px;
}

.pe-box {
    border: 1px solid #c8d0dc;
    border-radius: 9px;
    padding: 9px 12px;
    box-sizing: border-box;
}

.pe-title {
    font-size: 16px;
    font-weight: bold;
    color: #243447;
    margin-bottom: 6px;
}

.pe-info {
    font-size: 13px;
    line-height: 1.52;
}

.pe-label {
    display: inline-block;
    min-width: 235px;
    font-weight: bold;
}

.pe-value {
    font-size: 14px;
    font-weight: bold;
}

.pe-note {
    font-size: 12px;
    line-height: 1.35;
    color: #555555;
    margin-top: 6px;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea {
    overflow-x: visible !important;
    max-width: none !important;
}

</style>
""")


# ------------------------------------------------------------
# Title
# ------------------------------------------------------------

title_html = HTML("""
<div class="pe-root">

    <div style="
        font-family:monospace;
        font-size:22px;
        font-weight:bold;
        margin-bottom:8px;
    ">
        Product Quantization Error: Decomposition and Periodicity
    </div>

</div>
""")


# ------------------------------------------------------------
# Visible introductory section
# ------------------------------------------------------------

description_html = HTML("""
<div class="pe-root">

    <div class="pe-description">

        The finite-precision product is formed through

        <div style="text-align:center; margin:7px 0;">
            <b>x → Q(x) → αQ(x) → Q(αQ(x)).</b>
        </div>

        Therefore,

        <div style="text-align:center; margin:7px 0;">
            <b>
            e<sub>total</sub>
            =
            e<sub>product</sub>
            +
            αe<sub>input</sub>.
            </b>
        </div>

        The first two display modes study this general decomposition using
        the coefficient <b>α</b>. The last two modes study the special
        rational-multiplier case <b>α = L/M</b> with the already-quantized
        input <b>x = nΔ</b>.

    </div>


    <div class="pe-observe">

        <div class="pe-observe-title">
            What we expect to observe
        </div>

        <b>Total product error:</b>
        changing α changes the error structure, while increasing K reduces
        the absolute quantization step Δ = 2<sup>−K</sup> and hence the
        physical magnitude of the errors.<br><br>

        <b>Error decomposition:</b>
        the total error is formed by adding the product-rounding error and
        the scaled input-quantization contribution αe<sub>input</sub>.
        Increasing |α| increases the importance of the latter term.<br><br>

        <b>Rational multiplier periodicity:</b>
        for α = L/M in lowest terms, the normalized rounding-error sequence
        repeats every <b>M samples</b>. Vertical dashed separators mark the
        successive periods.<br><br>

        <b>One-period view:</b>
        the discrete error values within a complete period can be examined
        directly and occur in multiples of <b>Δ/M</b>.

    </div>


    <div class="pe-normalization">

        <div class="pe-normalization-title">
            Important note about K and normalization
        </div>

        In the rational-multiplier plots, the vertical variable is
        <b>e/Δ</b>. Since both the real error and Δ scale together when
        K changes, this normalization removes the visible dependence on
        K from the shape of the curve.<br><br>

        The physical error still decreases when K increases because
        <b>Δ = 2<sup>−K</sup></b> becomes smaller. The K control is therefore
        disabled in the normalized rational-multiplier plots to avoid giving
        the impression that it should alter their shape.

    </div>

</div>
""")


# ------------------------------------------------------------
# Dynamic summary
# ------------------------------------------------------------

summary_html = HTML()

summary_html.layout = Layout(width='610px', min_width='610px', overflow='visible')


# ------------------------------------------------------------
# Main interactive function
# ------------------------------------------------------------

def plot_product_error(display_mode='Total product error', alpha=0.70, K=4, L=5, M=16, number_of_samples=64):

    Delta = 2.0**(-K)

    divisor = math.gcd(L, M)

    Lr = L // divisor

    Mr = M // divisor

    alpha_rational = Lr / Mr

    x = np.linspace(-4.0 * Delta, 4.0 * Delta, 4001)

    qx, q_product, e_input, e_product, e_input_contribution, e_total = product_error_components(x, alpha, Delta)

    total_bound = (1.0 + abs(alpha)) * Delta / 2.0

    n_r, error_r, Delta_r, alpha_r, Lrr, Mrr = rational_error_sequence(L, M, K, max(number_of_samples, 2 * Mr))

    if len(error_r) >= 2 * Mrr:

        periodicity_difference = np.max(np.abs(error_r[:Mrr] - error_r[Mrr:2 * Mrr]))

    else:

        periodicity_difference = np.nan


    # --------------------------------------------------------
    # Dynamic information panel
    # --------------------------------------------------------

    if display_mode in ['Total product error', 'Error decomposition']:

        mode_information = f"""
        <span class="pe-label">Active model</span>
        General multiplier α
        <br>

        <span class="pe-label">General multiplier</span>
        α = <span class="pe-value">{alpha:.4f}</span>
        <br>

        <span class="pe-label">Fractional bits</span>
        K = <span class="pe-value">{K}</span>
        <br>

        <span class="pe-label">Quantization step</span>
        Δ = {Delta:.8f}
        <br>

        <span class="pe-label">Product-error bound</span>
        |e<sub>total</sub>| ≤ {total_bound:.8e}
        """

    else:

        mode_information = f"""
        <span class="pe-label">Active model</span>
        Rational multiplier α = L/M
        <br>

        <span class="pe-label">Selected ratio</span>
        {L}/{M}
        <br>

        <span class="pe-label">Reduced ratio</span>
        {Lr}/{Mr}
        <br>

        <span class="pe-label">Rational α</span>
        {alpha_rational:.8f}
        <br>

        <span class="pe-label">Predicted period</span>
        <span class="pe-value">{Mr} samples</span>
        <br>

        <span class="pe-label">Normalized level spacing</span>
        1/M = {1.0 / Mr:.8f}
        <br>

        <span class="pe-label">Physical level spacing</span>
        Δ/M = {Delta / Mr:.8e}
        <br>

        <span class="pe-label">Periodicity check</span>
        max |e[n+M]−e[n]| = {periodicity_difference:.3e}
        """


    summary_html.value = f"""
    <div class="pe-box">

        <div class="pe-title">
            Current Product-Quantization Data
        </div>

        <div class="pe-info">

            <span class="pe-label">Selected display</span>
            {display_mode}
            <br>

            {mode_information}

        </div>

        <div class="pe-note">
            Controls that do not affect the selected representation are
            automatically disabled.
        </div>

    </div>
    """


    # --------------------------------------------------------
    # Figure
    # --------------------------------------------------------

    fig, ax = plt.subplots(figsize=(12.0, 5.0))


    # ========================================================
    # TOTAL PRODUCT ERROR
    # ========================================================

    if display_mode == 'Total product error':

        ax.plot(x, e_total, linewidth=1.8, label='Total product error')

        ax.axhline(total_bound, linestyle='--', linewidth=1.2, label='Conservative error bound')

        ax.axhline(-total_bound, linestyle='--', linewidth=1.2)

        ax.axhline(0.0, linewidth=0.8)

        ax.set_xlim(x[0], x[-1])

        ax.set_xlabel('Input x')

        ax.set_ylabel('Actual total error e')

        ax.set_title('Total Error in the Quantized Product', fontsize=12)

        ax.grid(True, linestyle=':', alpha=0.5)

        ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.17), ncol=2, frameon=False, fontsize=9)


    # ========================================================
    # ERROR DECOMPOSITION
    # ========================================================

    elif display_mode == 'Error decomposition':

        ax.plot(x, e_product, linewidth=1.7, label='Product-rounding error')

        ax.plot(x, e_input_contribution, linestyle='--', linewidth=1.7, label='α × input-quantization error')

        ax.plot(x, e_total, linestyle='-.', linewidth=2.0, label='Total error')

        ax.axhline(0.0, linewidth=0.8)

        ax.set_xlim(x[0], x[-1])

        ax.set_xlabel('Input x')

        ax.set_ylabel('Actual error')

        ax.set_title('Decomposition of the Product Quantization Error', fontsize=12)

        ax.grid(True, linestyle=':', alpha=0.5)

        ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.17), ncol=3, frameon=False, fontsize=9)


    # ========================================================
    # RATIONAL MULTIPLIER PERIODICITY
    # ========================================================

    elif display_mode == 'Rational multiplier periodicity':

        n, error, Delta_used, alpha_used, L_used, M_used = rational_error_sequence(L, M, K, number_of_samples)

        ax.plot(n, error / Delta_used, 'o-', linewidth=1.4, markersize=4, label=f'α = {L_used}/{M_used}')

        separator = M_used

        while separator < number_of_samples:

            ax.axvline(separator - 0.5, linestyle='--', linewidth=1.0, alpha=0.7)

            separator += M_used

        ax.axhline(0.0, linewidth=0.8)

        ax.set_xlim(-0.5, number_of_samples - 0.5)

        ax.set_xlabel('Input index n')

        ax.set_ylabel('Normalized product-rounding error e / Δ')

        ax.set_title(f'Periodic Rounding Error for α = {L_used}/{M_used}  —  Period M = {M_used}', fontsize=12)

        ax.grid(True, linestyle=':', alpha=0.5)

        ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.17), ncol=1, frameon=False, fontsize=9)


    # ========================================================
    # ERROR VALUES OVER ONE PERIOD
    # ========================================================

    else:

        n, error, integer_levels, Delta_used, alpha_used, L_used, M_used = one_period_error(L, M, K)

        markerline, stemlines, baseline = ax.stem(n, error / Delta_used, basefmt=' ', label=f'One period, α = {L_used}/{M_used}')

        plt.setp(stemlines, linewidth=1.5)

        plt.setp(markerline, markersize=6)

        ax.axhline(0.0, linewidth=0.8)

        ax.set_xlim(-0.5, M_used - 0.5)

        ax.set_xticks(n)

        ax.set_xlabel('n within one period')

        ax.set_ylabel('Normalized error e / Δ')

        ax.set_title(f'Discrete Rounding-Error Values over One Period  (M = {M_used})', fontsize=12)

        ax.grid(True, linestyle=':', alpha=0.5)

        for ni, ei, ell in zip(n, error / Delta_used, integer_levels):

            ax.text(ni, ei + 0.035, f'ℓ={ell}', ha='center', va='bottom', fontsize=8)

        ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.17), ncol=1, frameon=False, fontsize=9)


    plt.subplots_adjust(left=0.08, right=0.98, top=0.90, bottom=0.25)

    plt.show()

    plt.close(fig)


# ------------------------------------------------------------
# Controls
# ------------------------------------------------------------

slider_layout = Layout(width='295px')

slider_style = {'description_width': '115px'}


display_selector = RadioButtons(
    options=[
        'Total product error',
        'Error decomposition',
        'Rational multiplier periodicity',
        'Error values over one period'
    ],
    value='Total product error',
    description='Display:',
    style={'description_width': '65px'},
    layout=Layout(width='300px')
)


alpha_slider = FloatSlider(
    value=0.70,
    min=0.10,
    max=3.00,
    step=0.05,
    description='General alpha:',
    continuous_update=True,
    style=slider_style,
    layout=slider_layout,
    readout_format='.2f'
)


K_slider = IntSlider(
    value=4,
    min=2,
    max=12,
    step=1,
    description='Bits K:',
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)


L_slider = IntSlider(
    value=5,
    min=1,
    max=31,
    step=1,
    description='Numerator L:',
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)


M_slider = IntSlider(
    value=16,
    min=2,
    max=32,
    step=1,
    description='Denominator M:',
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)


samples_slider = IntSlider(
    value=64,
    min=16,
    max=160,
    step=4,
    description='Samples:',
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)


# ------------------------------------------------------------
# Enable only controls relevant to the selected display
# ------------------------------------------------------------

def update_control_states(change=None):

    mode = display_selector.value


    if mode == 'Total product error':

        alpha_slider.disabled = False

        K_slider.disabled = False

        L_slider.disabled = True

        M_slider.disabled = True

        samples_slider.disabled = True


    elif mode == 'Error decomposition':

        alpha_slider.disabled = False

        K_slider.disabled = False

        L_slider.disabled = True

        M_slider.disabled = True

        samples_slider.disabled = True


    elif mode == 'Rational multiplier periodicity':

        alpha_slider.disabled = True

        K_slider.disabled = True

        L_slider.disabled = False

        M_slider.disabled = False

        samples_slider.disabled = False


    else:

        alpha_slider.disabled = True

        K_slider.disabled = True

        L_slider.disabled = False

        M_slider.disabled = False

        samples_slider.disabled = True


display_selector.observe(update_control_states, names='value')


# ------------------------------------------------------------
# Set initial control state
# ------------------------------------------------------------

update_control_states()


# ------------------------------------------------------------
# Interactive object
# ------------------------------------------------------------

widget_plot = interactive(
    plot_product_error,
    display_mode=display_selector,
    alpha=alpha_slider,
    K=K_slider,
    L=L_slider,
    M=M_slider,
    number_of_samples=samples_slider
)


# ------------------------------------------------------------
# Controls box
# ------------------------------------------------------------

controls_box = VBox(
    [
        HTML("<div class='pe-title'>Controls</div>"),
        display_selector,
        alpha_slider,
        K_slider,
        L_slider,
        M_slider,
        samples_slider
    ],
    layout=Layout(
        width='335px',
        min_width='335px',
        border='1px solid #c8d0dc',
        padding='9px',
        overflow='visible',
        align_items='flex-start'
    )
)


# ------------------------------------------------------------
# Summary + controls
# ------------------------------------------------------------

top_row = HBox(
    [
        summary_html,
        controls_box
    ],
    layout=Layout(
        width='960px',
        max_width='960px',
        overflow='visible',
        align_items='flex-start',
        justify_content='space-between'
    )
)


# ------------------------------------------------------------
# Plot output
# ------------------------------------------------------------

plot_output = widget_plot.children[-1]

plot_output.layout = Layout(width='auto', overflow='visible')


# ------------------------------------------------------------
# Final layout
# ------------------------------------------------------------

main_layout = VBox(
    [
        description_html,
        top_row,
        plot_output
    ],
    layout=Layout(
        width='960px',
        overflow='visible',
        align_items='flex-start'
    )
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

display(style_html)

display(title_html)

display(main_layout)